# Week 4: Dimensionality Reduction with PCA

This notebook demonstrates dimensionality reduction using Principal Component Analysis (PCA) on the Breast Cancer dataset from scikit-learn. We'll compare the performance of Random Forest classifier on:
1. Raw data
2. Standardized data
3. PCA-transformed data (retaining 95% variance)

In [2]:
# Import necessary libraries
import pandas as pd
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif, RFE
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import sklearn.datasets as DS

## Helper Function for Random Forest Classification

We'll create a helper function to train a Random Forest classifier and evaluate its performance.

In [3]:
def RF_Classs(X, Y, XS, YS):
    """
    Train a Random Forest classifier and return accuracy score
    
    Parameters:
    X: Training features
    Y: Training labels
    XS: Test features
    YS: Test labels
    """
    RF = RandomForestClassifier(random_state=42)
    RF.fit(X, Y)
    PR = RF.predict(XS)
    accuracy = accuracy_score(PR, YS)
    print(f"Accuracy: {accuracy:.4f}")
    return accuracy

## Load and Prepare the Dataset

Load the Breast Cancer dataset and split it into training and testing sets.

In [ ]:
# Load the breast cancer dataset
Data = DS.load_breast_cancer()
X = Data['data']
Y = Data['target']

print(f"Dataset shape: {X.shape}")
print(f"Number of features: {X.shape[1]}")
print(f"Number of samples: {X.shape[0]}")
print(f"Feature names: {Data.feature_names[:5]}...")  # Show first 5 feature names

In [ ]:
# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, Y, train_size=0.3, random_state=42)

print(f"Training set shape: {X_train.shape}")
print(f"Test set shape: {X_test.shape}")

## Performance on Raw Data

First, let's evaluate the Random Forest classifier on the raw, unprocessed data.

In [ ]:
print('Result of raw data:')
raw_accuracy = RF_Classs(X_train, y_train, X_test, y_test)

## Standardize the Data

Standardize the features to have zero mean and unit variance. This is important for PCA as it's sensitive to the scale of the features.

In [ ]:
# Apply StandardScaler
Sc = StandardScaler()
X_train_S = Sc.fit_transform(X_train)
X_test_S = Sc.transform(X_test)

print(f"Standardized training data mean: {np.mean(X_train_S, axis=0)[:5]}")  # Should be close to 0
print(f"Standardized training data std: {np.std(X_train_S, axis=0)[:5]}")   # Should be close to 1

## Performance on Standardized Data

Evaluate the Random Forest classifier on the standardized data.

In [ ]:
print('Result of scaled data:')
scaled_accuracy = RF_Classs(X_train_S, y_train, X_test_S, y_test)

## Apply PCA (Retaining 95% Variance)

Apply Principal Component Analysis to reduce dimensionality while retaining 95% of the variance in the data.

In [ ]:
# Initialize PCA to retain 95% of variance
P1 = PCA(n_components=0.95)

# Fit and transform the training data
XPCA1 = P1.fit_transform(X_train_S)
XPCAS = P1.transform(X_test_S)

# Record the number of components retained by PCA
n_components_retained = P1.n_components_
print(f'Number of components retained by PCA (95% variance): {n_components_retained}')
print(f'Original number of features: {X_train_S.shape[1]}')
print(f'Dimensionality reduction: {X_train_S.shape[1]} → {n_components_retained}')
print(f'Reduction ratio: {(1 - n_components_retained/X_train_S.shape[1]):.2%}')

## Analyze PCA Components

Let's examine the variance explained by each principal component.

In [ ]:
print(f'Variance explained by each component:')
for i, ratio in enumerate(P1.explained_variance_ratio_):
    print(f'  Component {i+1}: {ratio:.4f} ({ratio*100:.2f}%)')

print(f'\nCumulative variance explained: {np.cumsum(P1.explained_variance_ratio_)[-1]:.4f}')
print(f'Total variance retained: {np.cumsum(P1.explained_variance_ratio_)[-1]*100:.2f}%')

## Performance on PCA-Transformed Data

Finally, evaluate the Random Forest classifier on the PCA-transformed data.

In [ ]:
print('Result of PCA data:')
pca_accuracy = RF_Classs(XPCA1, y_train, XPCAS, y_test)

## Summary of Results

Compare the performance across different data preprocessing approaches.

In [ ]:
print("\n" + "="*50)
print("SUMMARY OF RESULTS")
print("="*50)
print(f"Raw data accuracy:          {raw_accuracy:.4f}")
print(f"Standardized data accuracy: {scaled_accuracy:.4f}")
print(f"PCA data accuracy:          {pca_accuracy:.4f}")
print("\nDimensionality Reduction Summary:")
print(f"Original features:          {X_train_S.shape[1]}")
print(f"PCA components (95% var):   {n_components_retained}")
print(f"Reduction achieved:         {(1 - n_components_retained/X_train_S.shape[1]):.2%}")
print(f"Variance retained:          {np.cumsum(P1.explained_variance_ratio_)[-1]*100:.2f}%")